# Microsoft Foundry · Agente com várias ferramentas (MCP + OpenAPI)

Laboratório do curso de **Microsoft Foundry** (Seção 4 · Aula 18): um único agente com **duas ferramentas**, o servidor **MCP do Microsoft Learn** e a **API do clima via OpenAPI**, para observar como o orquestrador interno do agente **planeja** e encaminha cada parte da pergunta para a ferramenta certa.

## Sumário

0. [Visão geral do projeto](#0.-Visão-geral-do-projeto)
1. [Instalação das bibliotecas](#1.-Instalação-das-bibliotecas)
2. [Variáveis de ambiente](#2.-Variáveis-de-ambiente)
3. [Conectando ao projeto do Foundry](#3.-Conectando-ao-projeto-do-Foundry)
4. [Cliente OpenAI do projeto](#4.-Cliente-OpenAI-do-projeto)
5. [Ferramenta 1: OpenAPI do clima](#5.-Ferramenta-1:-OpenAPI-do-clima)
6. [Ferramenta 2: servidor MCP do Microsoft Learn](#6.-Ferramenta-2:-servidor-MCP-do-Microsoft-Learn)
7. [Criando o agente com várias ferramentas](#7.-Criando-o-agente-com-várias-ferramentas)
8. [Criando a conversa](#8.-Criando-a-conversa)
9. [Perguntando ao agente](#9.-Perguntando-ao-agente)
10. [Verificando no portal do Foundry](#10.-Verificando-no-portal-do-Foundry)
11. [Resumo dos aprendizados](#11.-Resumo-dos-aprendizados)
12. [Limpeza e custos](#12.-Limpeza-e-custos)
13. [Interface com Streamlit](#13.-Interface-com-Streamlit)

---
## 0. Visão geral do projeto

### 0.1 Problema

Nos labs anteriores, cada agente tinha **uma** ferramenta. Na prática, um assistente precisa combinar várias fontes (APIs, documentação, bases internas) e uma mesma pergunta pode exigir mais de uma delas. Programar à mão "se a pergunta fala de X, chame Y" não escala.

### 0.2 Objetivo

- Reaproveitar as ferramentas dos Labs 16 (**OpenAPI do clima**) e 17 (**MCP do Microsoft Learn**) em um só agente.
- Fazer uma pergunta com **duas intenções** e verificar se o agente monta um **plano de execução** dinâmico (ferramenta 1, depois ferramenta 2).
- Conferir no **tracing** quais ferramentas foram chamadas, em que ordem e com quais argumentos.

### 0.3 Ferramentas e serviços da Azure

| Serviço / ferramenta | Papel no projeto |
| --- | --- |
| **Microsoft Foundry** (projeto) | Conexão MCP registrada, agente, conversas e traces. |
| **Foundry Agent Service** | Hospeda o `multi-tool-agent` e **orquestra** as ferramentas. |
| **`MCPTool`** + conexão do projeto | Acesso ao servidor MCP do Microsoft Learn (`https://learn.microsoft.com/api/mcp`). |
| **Ferramenta OpenAPI** | Acesso à API pública de clima **wttr.in** pela especificação `weather_openapi.json`. |
| **Azure OpenAI** (deployment GPT-4) | Planeja, escolhe as ferramentas e combina os resultados. |
| **Microsoft Entra ID** | Autenticação sem chave (`DefaultAzureCredential`). |

### 0.4 Como funciona (visão simples)

![Fluxo do agente com várias ferramentas](fluxo_multi_tool.svg)

### 0.5 Arquitetura

![Arquitetura do projeto](arquitetura.svg)

> 💡 **Para um leigo:** é como um atendente com dois telefones: um liga para a "central do clima" e o outro para a "biblioteca da Microsoft". Diante de um pedido duplo, ele faz as duas ligações e responde tudo de uma vez.

> ⚠️ **Pré-requisito:** a conexão MCP do Microsoft Learn precisa estar registrada no portal (passo a passo no **Lab 17**, seção 1). O nome dela vai em `MCP_SERVER_NAME`.

---
## 1. Instalação das bibliotecas

| Biblioteca | Para que serve |
| --- | --- |
| `azure-ai-projects==2.0.0b2` | **Foundry SDK v2**: cliente do projeto (`AIProjectClient`), agentes e ferramentas. Versão *beta*, fixada para o código funcionar igual ao da aula. |
| `openai==1.109.1` | Cliente OpenAI, usado para criar conversas e enviar mensagens ao agente (Responses API). |
| `python-dotenv` | Lê o arquivo `.env` com as configurações. |
| `azure-identity` | Autenticação na Azure sem chave no código (`DefaultAzureCredential`). |
| `jsonref` | Lê a especificação OpenAPI resolvendo os `$ref`. ⚠️ É importada no notebook, mas **não está** no `%pip install` abaixo: se ainda não estiver instalada, rode `%pip install jsonref`. |

> ⚠️ Versões fixadas (`==`) evitam surpresas: SDKs em *beta* mudam nomes de classes e métodos entre versões.

In [ ]:
# Instalando as bibliotecas no mesmo ambiente do kernel do notebook (%pip)
# azure-ai-projects = Foundry SDK | openai = conversa com o agente
# python-dotenv = lê o .env | azure-identity = login na Azure sem chave no código
%pip install azure-ai-projects==2.0.0b2 openai==1.109.1 python-dotenv azure-identity

---
## 2. Variáveis de ambiente

As configurações ficam no arquivo **`.env`** (fora do código):

| Variável | O que é | Onde encontrar |
| --- | --- | --- |
| `FOUNDRY_PROJECT_ENDPOINT` | Endereço (URL) do projeto Foundry. | Portal do Foundry → página **Início** (*Overview*) → *Project endpoint*. |
| `MODEL_DEPLOYMENT_NAME` | Nome do deployment do modelo de chat. | Portal do Foundry → **Modelos + endpoints**. |
| `MCP_SERVER_NAME` | Nome da conexão MCP do Microsoft Learn. | Portal do Foundry → **Tools** (registrada no Lab 17). |

```
FOUNDRY_PROJECT_ENDPOINT=https://<seu-recurso>.services.ai.azure.com/api/projects/<seu-projeto>
MODEL_DEPLOYMENT_NAME=<nome-do-deployment>
MCP_SERVER_NAME=<mcp-server-name>
```

> 🔒 **Segurança:** o `.env` guarda apenas **endereços e nomes** (nenhuma chave), porque a autenticação é feita pelo Microsoft Entra ID (seção seguinte). Se algum dia ele passar a ter segredos (chaves de API, tokens), não deve ser publicado: nesse caso, adicione `.env` ao `.gitignore`.

In [ ]:
# Importando as bibliotecas e lendo as configurações do arquivo .env
# os.getenv busca cada valor pelo nome; se a variável não existir, retorna None
# jsonref = lê a spec OpenAPI | MCPTool = ferramenta MCP | MCP_SERVER_NAME = conexão do portal
import os
from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
import jsonref
from azure.ai.projects.models import PromptAgentDefinition, MCPTool, Tool

load_dotenv()

foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
mcp_server_name = os.getenv("MCP_SERVER_NAME")

---
## 3. Conectando ao projeto do Foundry

O **`AIProjectClient`** é a porta de entrada do Python para o projeto no Foundry: por ele criamos agentes, listamos conexões e obtemos o cliente OpenAI.

- **`endpoint`**: o endereço do projeto (do `.env`).
- **`credential`**: o **`DefaultAzureCredential`** reaproveita um login que já existe na máquina (Azure CLI `az login`, conta Azure do VS Code, *Managed Identity* quando roda na Azure…). Assim, **nenhuma chave fica escrita no código**: quem autentica é o **Microsoft Entra ID**.

> 💡 Para rodar localmente, basta executar `az login` antes. O usuário precisa ter permissão no projeto (por exemplo, a função **Azure AI User**).

In [ ]:
# Criando o cliente do projeto Foundry: endpoint do .env + login da máquina (Entra ID)
# A conexão é "preguiçosa": erros de login só aparecem na primeira chamada real
project_client = AIProjectClient(
    endpoint=foundry_project_endpoint,
    credential=DefaultAzureCredential(),
)

---
## 4. Cliente OpenAI do projeto

O agente roda sobre um modelo **GPT** (Azure OpenAI). Para conversar com ele usamos a **Responses API** do SDK `openai`, e o próprio projeto entrega esse cliente já configurado com o endpoint e a autenticação do Foundry: `project_client.get_openai_client()`.

In [ ]:
# Obtendo o cliente OpenAI já autenticado no projeto Foundry
# É ele que cria conversas e envia mensagens ao agente (Responses API)
openai_client = project_client.get_openai_client()

---
## 5. Ferramenta 1: OpenAPI do clima

Igual ao Lab 16: a especificação `weather_openapi.json` descreve a API **wttr.in** (`GET /{location}?format=j1`, operação `GetCurrentWeather`), e a ferramenta é um dicionário:

| Chave | Valor |
| --- | --- |
| `type` | `"openapi"` |
| `openapi.name` | `"weather"` |
| `openapi.spec` | conteúdo do JSON |
| `openapi.auth.type` | `"anonymous"` (API pública) |

In [ ]:
# Ferramenta 1: lendo a especificação OpenAPI e montando a ferramenta do clima
with open("./weather_openapi.json", "r") as f:
        openapi_weather = jsonref.loads(f.read())

# Initialize agent OpenApi tool using the read in OpenAPI spec
weather_tool = {
        "type": "openapi",
        "openapi":{
            "name": "weather",
            "spec": openapi_weather,
            "auth": {
                "type": "anonymous"
            },
        }
}

---
## 6. Ferramenta 2: servidor MCP do Microsoft Learn

Igual ao Lab 17, em dois passos:

1. **Obter o ID da conexão**: percorrer `project_client.connections.list()` e guardar o `id` da conexão chamada `MCP_SERVER_NAME`.
2. **Montar o `MCPTool`**: `server_label="microsoft_learn_server"`, `server_url="https://learn.microsoft.com/api/mcp"`, `require_approval="never"` (o agente executa sem pedir aprovação) e `project_connection_id=connection_id`.

In [ ]:
# Ferramenta 2 (passo 1): procurando o ID da conexão MCP pelo nome
connection_id = ""

for connection in project_client.connections.list():
    if connection.name == mcp_server_name:
        connection_id = connection.id
        break

print(f"The MCP Server Connection ID is: {connection_id}")

In [ ]:
# Ferramenta 2 (passo 2): montando o MCPTool do Microsoft Learn
mcp_tool = MCPTool(
    server_label = "microsoft_learn_server",
    server_url="https://learn.microsoft.com/api/mcp",
    require_approval="never",
    project_connection_id=connection_id
)

---
## 7. Criando o agente com várias ferramentas

O ponto central do lab: para dar **várias ferramentas** a um agente, basta colocá-las **na mesma lista** `tools`.

| Campo | Valor |
| --- | --- |
| `agent_name` | `multi-tool-agent` |
| `instructions` | `"You are a helpful assistant that can use multiple tools to answer user queries."` → *"Você é um assistente útil que pode usar várias ferramentas para responder às perguntas do usuário."* |
| `tools` | `[weather_tool, mcp_tool]` |

> 💡 Os tipos podem ser misturados (OpenAPI, MCP, Web Search, Code Interpreter…). O agente lê a descrição de cada uma para decidir quando usá-la.

In [ ]:
# Criando o agente com as DUAS ferramentas na lista tools
agent_name = "multi-tool-agent"

agent = project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are a helpful assistant that can use multiple tools to answer user queries.",
        tools=[weather_tool, mcp_tool],
    )
)

print(f"Created agent: {agent.name} with ID: {agent.id}")

---
## 8. Criando a conversa

A **conversation** é um objeto guardado na Azure que **armazena o histórico** entre o usuário e o agente, identificado pelo `conversation.id`. Todas as mensagens enviadas com o mesmo id fazem parte do mesmo diálogo, então o agente "lembra" o que já foi dito.

> 💡 No portal, o mesmo `conversation.id` aparece em **Traces**, com cada chamada de ferramenta e o consumo de tokens.

In [ ]:
# Criando a conversa que guarda o histórico (memória) entre usuário e agente
# create a conversation to use with the agent
conversation = openai_client.conversations.create()
print(f"Created conversation with id: {conversation.id}")

---
## 9. Perguntando ao agente

A pergunta tem **duas partes**, uma para cada ferramenta: *"Me diga o clima em Seattle e também encontre um módulo do Microsoft Learn sobre o Microsoft Foundry."*

In [ ]:
# Pergunta com duas intenções: clima (OpenAPI) + módulo do Learn (MCP)
user_query = "Let me know the weather in Seattle and also find me a Microsoft Learn module on Microsoft Foundry."

| Parâmetro | Valor | Para que serve |
| --- | --- | --- |
| `conversation` | `conversation.id` | Grava a mensagem na conversa e envia o histórico ao modelo. |
| `extra_body` → `agent` | `{"name": "multi-tool-agent", "type": "agent_reference"}` | Diz **qual agente** responde. Sem versão informada, usa **a mais recente**. |
| `input` | pergunta do usuário | O texto enviado ao agente. |

A resposta final em texto fica em **`response.output_text`**. Repare que a pergunta **não manda** o agente usar a ferramenta: é o próprio agente que analisa a intenção e decide.

In [ ]:
# Enviando a pergunta: o agente planeja, chama as duas ferramentas e combina os resultados
response = openai_client.responses.create(
    conversation=conversation.id,
    extra_body = {
        "agent": {
            "name": agent_name,
            "type": "agent_reference"
        }
    },
    input = user_query
)

print(f"Agent response: {response.output_text}")

### 9.1 O que aconteceu na demonstração da aula

| Parte da pergunta | Ferramenta usada | Resultado |
| --- | --- | --- |
| Clima em Seattle | OpenAPI `GetCurrentWeather` | 7 °C, nublado, umidade 82%, vento sul de 3 km/h, visibilidade 16 km, UV 0. |
| Módulo sobre o Microsoft Foundry | MCP `microsoft_docs_search` | Módulo **"What is Microsoft Foundry"**, com destaques e link para o conteúdo completo no Microsoft Learn. |

✅ Uma pergunta, duas ferramentas, uma resposta única: o agente fez o roteamento sozinho.

---
## 10. Verificando no portal do Foundry

Em **Agentes → `multi-tool-agent`** aparecem as **duas ferramentas**. Em **Traces**, abrindo o `conversation.id`:

| Ordem | Chamada | Detalhe |
| --- | --- | --- |
| 1 | `mcp_list_tools` | O agente pediu ao servidor MCP a lista de ferramentas. |
| 2 | `mcp_call` → `microsoft_docs_search` | Busca de módulos sobre o Microsoft Foundry no learn.microsoft.com. |
| 3 | chamada remota → `GetCurrentWeather` | Clima de Seattle pela especificação OpenAPI. |

Total de cerca de **17.000 tokens**: o conteúdo do Learn e o JSON do clima entram no contexto.

---
## 11. Resumo dos aprendizados

- Várias ferramentas = **várias entradas na lista `tools`** do `PromptAgentDefinition`.
- O Foundry Agent Service tem um **orquestrador interno**: o agente divide a pergunta, monta um plano e escolhe a ferramenta de cada parte.
- Ferramentas de tipos diferentes (MCP + OpenAPI) convivem no mesmo agente.
- O **tracing** mostra a ordem real de execução e os argumentos de cada chamada.
- Mais ferramentas e mais resultados = **mais tokens**: vale monitorar o custo por pergunta.

---
## 12. Limpeza e custos

| Recurso | Gera custo? |
| --- | --- |
| Chamadas ao modelo (tokens) | Sim, **por uso**: só quando você conversa com o agente. Ferramentas que trazem muito texto (páginas, APIs, documentos) aumentam bastante os tokens. |
| Servidor MCP e API wttr.in | Gratuitos e públicos (sem custo na Azure). |

Ao terminar os estudos:

- **Portal do Foundry → Agentes**: apagar as versões do agente que não serão mais usadas.
- Ao final do curso, apagar o **grupo de recursos** no `portal.azure.com` remove tudo de uma vez.

> 💡 Vale criar um **orçamento com alerta** em *Cost Management → Budgets* para ser avisado de qualquer gasto inesperado.

---
## 13. Interface com Streamlit

A célula abaixo gera o arquivo **`app.py`**: o mesmo agente, agora com uma interface de chat no navegador. Ela reaproveita o código do notebook:

| Função do `app.py` | Código do notebook reaproveitado |
| --- | --- |
| `criar_clientes()` | `AIProjectClient` (seção 3) e `get_openai_client()` (seção 4) |
| `nova_conversa() / perguntar()` | `conversations.create` (seção 8) e `responses.create` (seção 9) |
| `plano_executado()` | lê as etapas da resposta (`mcp_list_tools`, `mcp_call`, chamada OpenAPI): o mesmo plano visto no tracing (seção 10) |

Pontos importantes:

- O Streamlit **reexecuta o script inteiro** a cada interação. Por isso os clientes ficam em cache (`@st.cache_resource`) e o `conversation.id` e o histórico ficam no `st.session_state`.
- O agente **não é recriado** a cada mensagem: o app usa o agente já criado neste notebook (pelo nome, sempre na versão mais recente).
- Abaixo de cada resposta aparece o **plano executado**: quais ferramentas o agente usou e em que ordem.
- A barra lateral tem **perguntas de teste**: uma só de clima, uma só do Learn e uma que usa as duas, para comparar o roteamento.

**Como executar:**

```bash
pip install streamlit
az login
streamlit run app.py
```

O app abre em `http://localhost:8501` e usa o mesmo arquivo `.env` desta pasta.

In [ ]:
%%writefile app.py

import os

import streamlit as st
from azure.ai.projects import AIProjectClient
from azure.core.exceptions import ClientAuthenticationError, HttpResponseError
from azure.identity import DefaultAzureCredential
from dotenv import load_dotenv
from openai import APIError

# ------------------------------------------------------------------
# 1. Configurações (as mesmas do notebook)
# ------------------------------------------------------------------
load_dotenv()

FOUNDRY_PROJECT_ENDPOINT = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
MODEL_DEPLOYMENT_NAME = os.getenv("MODEL_DEPLOYMENT_NAME")
AGENT_NAME = "multi-tool-agent"  # criado na seção 7 do notebook

st.set_page_config(page_title="Multi-Tool Agent · Microsoft Foundry", page_icon="🧰", layout="wide")


# ------------------------------------------------------------------
# 2. Conexão com o projeto do Foundry
# ------------------------------------------------------------------
@st.cache_resource
def criar_clientes():
    project_client = AIProjectClient(
        endpoint=FOUNDRY_PROJECT_ENDPOINT,
        credential=DefaultAzureCredential(),
    )
    return project_client.get_openai_client()


# ------------------------------------------------------------------
# 3. Conversa e pergunta
# ------------------------------------------------------------------
def nova_conversa():
    st.session_state.conversation_id = criar_clientes().conversations.create().id
    st.session_state.messages = []


def perguntar(texto):
    return criar_clientes().responses.create(
        conversation=st.session_state.conversation_id,
        extra_body={"agent": {"name": AGENT_NAME, "type": "agent_reference"}},
        input=texto,
    )


# ------------------------------------------------------------------
# 4. Plano de execução: quais ferramentas o agente usou e em que ordem
# ------------------------------------------------------------------
def plano_executado(response):
    etapas = []
    for item in response.output:
        if item.type == "message":
            continue
        if item.type == "mcp_list_tools":
            etapas.append("📋 MCP: listou as ferramentas do servidor")
        elif item.type == "mcp_call":
            etapas.append(f"🛰️ MCP: **{item.name}** `{item.arguments}`")
        else:
            # demais ferramentas (ex.: a chamada OpenAPI do clima)
            etapas.append(f"🔧 {item.type} {getattr(item, 'name', '') or ''}".strip())
    return etapas


def mostrar_plano(etapas):
    if etapas:
        with st.expander(f"🧭 Plano executado pelo agente ({len(etapas)} etapas)"):
            for numero, etapa in enumerate(etapas, start=1):
                st.markdown(f"{numero}. {etapa}")


# ------------------------------------------------------------------
# 5. Interface
# ------------------------------------------------------------------
st.title("🧰 Agente multiferramenta")
st.caption("Um agente, duas ferramentas (MCP do Microsoft Learn + API do clima via OpenAPI): ele decide quais usar e em que ordem.")

if not FOUNDRY_PROJECT_ENDPOINT or not MODEL_DEPLOYMENT_NAME:
    st.error("Configure FOUNDRY_PROJECT_ENDPOINT e MODEL_DEPLOYMENT_NAME no arquivo `.env`.")
    st.stop()

if "conversation_id" not in st.session_state:
    try:
        nova_conversa()
    except ClientAuthenticationError:
        st.error("Falha no login da Azure. Execute `az login` no terminal e recarregue a página.")
        st.stop()
    except (HttpResponseError, APIError) as erro:
        st.error(f"Erro ao criar a conversa no Foundry: {erro}")
        st.stop()

with st.sidebar:
    st.header("🧰 Ferramentas do agente")
    st.markdown("- 🌦️ **OpenAPI weather** → wttr.in\n- 📚 **MCP Microsoft Learn** → learn.microsoft.com")
    st.divider()
    st.caption(f"Agente: `{AGENT_NAME}`  \nConversa: `{st.session_state.conversation_id}`")
    if st.button("🗑️ Nova conversa"):
        nova_conversa()
        st.rerun()

    # Botões de exemplo: um para cada ferramenta e um que usa as duas
    st.divider()
    st.markdown("**Teste rápido**")
    exemplos = {
        "Só clima": "Como está o tempo em Lisboa hoje?",
        "Só Learn": "Encontre um módulo do Microsoft Learn sobre Azure AI Search.",
        "Os dois": "Let me know the weather in Seattle and also find me a Microsoft Learn module on Microsoft Foundry.",
    }
    for rotulo, texto in exemplos.items():
        if st.button(rotulo, use_container_width=True):
            st.session_state.pergunta_exemplo = texto

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])
        mostrar_plano(message.get("etapas"))

prompt = st.chat_input("Pergunte sobre clima, documentação Microsoft ou os dois") or st.session_state.pop("pergunta_exemplo", None)
if prompt:
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    with st.chat_message("assistant"):
        with st.spinner("Planejando e chamando as ferramentas..."):
            try:
                response = perguntar(prompt)
            except ClientAuthenticationError:
                st.error("Falha no login da Azure. Execute `az login` no terminal.")
            except (HttpResponseError, APIError) as erro:
                st.error(f"Erro ao consultar o agente: {erro}")
            else:
                etapas = plano_executado(response)
                st.markdown(response.output_text)
                mostrar_plano(etapas)
                st.session_state.messages.append({"role": "assistant", "content": response.output_text, "etapas": etapas})
